# 05 - Feedback Generation and Few-Shot Learning

This notebook generates structured resume feedback using the free open-source `Qwen/Qwen2.5-1.5B-Instruct` model. It does not require an API key.

Before running, select **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Install required libraries
!pip -q install transformers accelerate sentencepiece pandas pyarrow

In [ ]:
# Mount Google Drive and create output folders
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_PATH = Path('/content/drive/MyDrive/AI-Resume-Screener')
OUTPUT_PATH = BASE_PATH / 'outputs'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print('Project path:', BASE_PATH)

In [ ]:
# Confirm that Colab GPU is available
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')

if not torch.cuda.is_available():
    print('Warning: generation will be slow. Enable a T4 GPU before continuing.')

## Load the Open-Source Feedback Model

The first run downloads the model. Use `Qwen/Qwen2.5-0.5B-Instruct` instead if Colab runs out of memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
# For lower memory usage, replace the line above with:
# MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

feedback_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

feedback_model.eval()
print('Loaded:', MODEL_NAME)

## Define Few-Shot Examples

These examples teach the model the expected output format and how to assess niche roles without retraining.

In [ ]:
FEW_SHOT_EXAMPLES = """
EXAMPLE 1
Job role: MLOps Engineer
Resume evidence: Python, Docker, Kubernetes, CI/CD, MLflow, and AWS projects.
Required skills: Python, Docker, Kubernetes, CI/CD, MLflow, AWS, Terraform.
Match score: 0.86
Expected feedback:
Overall Match: Strong match with clear deployment and automation experience.
Strengths: Strong evidence in containerization, orchestration, CI/CD, and cloud deployment.
Matched Skills: Python, Docker, Kubernetes, CI/CD, MLflow, AWS.
Missing Skills: Terraform is not explicitly demonstrated.
Recommendations: Add evidence of infrastructure-as-code work or a Terraform project.
Limitations: The assessment only checks evidence stated in the resume.

EXAMPLE 2
Job role: Quantum Computing Researcher
Resume evidence: Physics PhD, Qiskit projects, quantum algorithms, and two publications.
Required skills: Qiskit, quantum algorithms, error correction, research publications.
Match score: 0.72
Expected feedback:
Overall Match: Good research alignment, but an important technical requirement is unclear.
Strengths: Strong academic background, Qiskit experience, and publications.
Matched Skills: Qiskit, quantum algorithms, research publications.
Missing Skills: Quantum error correction is not explicitly demonstrated.
Recommendations: Highlight any coursework, experiments, or publications involving error correction.
Limitations: Research quality and practical ability require human review.
"""

In [ ]:
# Build prompts and generate feedback
def build_messages(
    resume,
    job_description,
    match_score,
    matched_skills,
    missing_skills,
    use_few_shot=True,
):
    examples = FEW_SHOT_EXAMPLES if use_few_shot else 'No examples are provided.'

    system_message = (
        'You are an HR screening assistant. Use only evidence explicitly stated '
        'in the supplied resume and job description. Do not invent qualifications, '
        'experience, or personal characteristics. Treat the score as a supporting '
        'signal, not a final hiring decision.'
    )

    user_message = f"""
Study these examples before assessing the new candidate:

{examples}

NEW CANDIDATE

JOB DESCRIPTION:
{str(job_description)[:5000]}

RESUME:
{str(resume)[:5000]}

MATCH SCORE:
{float(match_score):.2f}

DETECTED MATCHED SKILLS:
{matched_skills}

DETECTED MISSING SKILLS:
{missing_skills}

Write concise feedback using exactly these headings:
Overall Match
Strengths
Matched Skills
Missing Skills
Recommendations
Limitations
"""

    return [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message},
    ]


def generate_feedback(
    resume,
    job_description,
    match_score,
    matched_skills,
    missing_skills,
    use_few_shot=True,
    max_new_tokens=450,
):
    messages = build_messages(
        resume,
        job_description,
        match_score,
        matched_skills,
        missing_skills,
        use_few_shot,
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=7000,
    ).to(feedback_model.device)

    with torch.no_grad():
        output = feedback_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Test Feedback Generation

In [ ]:
sample_resume = """
Machine learning engineer with experience building NLP applications using
Python and PyTorch. Deployed models using Docker and worked with SQL databases.
"""

sample_jd = """
We are hiring an NLP engineer with Python, PyTorch, transformers, Docker,
Kubernetes, SQL, and AWS experience.
"""

sample_feedback = generate_feedback(
    resume=sample_resume,
    job_description=sample_jd,
    match_score=0.72,
    matched_skills=['python', 'pytorch', 'docker', 'sql'],
    missing_skills=['kubernetes', 'aws'],
    use_few_shot=True,
)

print(sample_feedback)

## Compare Zero-Shot and Few-Shot Outputs

In [ ]:
zero_shot_feedback = generate_feedback(
    sample_resume,
    sample_jd,
    0.72,
    ['python', 'pytorch', 'docker', 'sql'],
    ['kubernetes', 'aws'],
    use_few_shot=False,
)

few_shot_feedback = generate_feedback(
    sample_resume,
    sample_jd,
    0.72,
    ['python', 'pytorch', 'docker', 'sql'],
    ['kubernetes', 'aws'],
    use_few_shot=True,
)

print('ZERO-SHOT OUTPUT\n')
print(zero_shot_feedback)
print('\n' + '=' * 80 + '\n')
print('FEW-SHOT OUTPUT\n')
print(few_shot_feedback)

## Generate Reports from Saved Project Predictions

This section uses `outputs/final_predictions.parquet` if Notebook 6 has already produced it. Otherwise, it tries `outputs/baseline_predictions.parquet`. Run this section after the appropriate predictions file exists.

In [ ]:
import ast
import pandas as pd

final_predictions_path = OUTPUT_PATH / 'final_predictions.parquet'
baseline_predictions_path = OUTPUT_PATH / 'baseline_predictions.parquet'

if final_predictions_path.exists():
    predictions_df = pd.read_parquet(final_predictions_path)
elif baseline_predictions_path.exists():
    predictions_df = pd.read_parquet(baseline_predictions_path)
else:
    predictions_df = None
    print('No predictions file found. Run the baseline/evaluation notebook first.')

if predictions_df is not None:
    print(predictions_df.shape)
    print(predictions_df.columns.tolist())
    display(predictions_df.head(2))

In [ ]:
# Lightweight skill extraction for report generation
import re

SKILLS = [
    'python', 'java', 'javascript', 'sql', 'mysql', 'postgresql',
    'mongodb', 'docker', 'kubernetes', 'aws', 'azure', 'gcp', 'git',
    'machine learning', 'deep learning', 'nlp', 'natural language processing',
    'tensorflow', 'pytorch', 'scikit-learn', 'react', 'node.js',
    'flask', 'django', 'fastapi', 'linux', 'excel', 'power bi', 'tableau',
]

def extract_skills(text):
    normalized = str(text).lower()
    return sorted({
        skill for skill in SKILLS
        if re.search(r'(?<!\\w)' + re.escape(skill) + r'(?!\\w)', normalized)
    })

def compare_skills(resume, jd):
    resume_skills = set(extract_skills(resume))
    jd_skills = set(extract_skills(jd))
    return sorted(resume_skills & jd_skills), sorted(jd_skills - resume_skills)

In [ ]:
# Generate a small report sample because local LLM generation takes time
REPORT_SAMPLE_SIZE = 10
generated_reports = []

if predictions_df is not None:
    report_sample = predictions_df.sample(
        n=min(REPORT_SAMPLE_SIZE, len(predictions_df)),
        random_state=42,
    ).reset_index(drop=True)

    for index, row in report_sample.iterrows():
        matched, missing = compare_skills(row['resume'], row['jd'])
        score = row['bert_score'] if 'bert_score' in row else row.get('tfidf_score', 0.0)

        report = generate_feedback(
            row['resume'],
            row['jd'],
            score,
            matched,
            missing,
            use_few_shot=True,
        )

        generated_reports.append({
            'sample_id': index,
            'true_label': row.get('label', ''),
            'match_score': float(score),
            'matched_skills': ', '.join(matched),
            'missing_skills': ', '.join(missing),
            'feedback': report,
        })

        print(f'Generated report {index + 1}/{len(report_sample)}')

In [ ]:
# Save generated reports and a human evaluation template
if generated_reports:
    reports_df = pd.DataFrame(generated_reports)
    reports_df.to_csv(OUTPUT_PATH / 'few_shot_feedback_reports.csv', index=False)

    evaluation_df = reports_df.copy()
    evaluation_df['correctness_1_to_5'] = ''
    evaluation_df['usefulness_1_to_5'] = ''
    evaluation_df['clarity_1_to_5'] = ''
    evaluation_df['hallucination_present_yes_no'] = ''
    evaluation_df['reviewer_comments'] = ''

    evaluation_df.to_csv(
        OUTPUT_PATH / 'feedback_human_evaluation_template.csv',
        index=False,
    )

    display(reports_df[['true_label', 'match_score', 'feedback']].head())
    print('Saved feedback reports and human evaluation template.')

## Report Notes

- Compare zero-shot and few-shot feedback for niche roles.
- Ask at least two human reviewers to score correctness, usefulness, clarity, and hallucination risk.
- State that the generated report supports human reviewers and must not make final hiring decisions.
- Mention that open-source model output quality depends on model size, prompt quality, and available GPU memory.